# Grant-proposal behavioural figures

One-off figures for the K99/R00 proposal, built on the tested `tmaze.analysis`
layer (ADR-0112) plus the new per-frame `tmaze.analysis.trajectories` helper
(ADR-0128). All logic lives in those modules; this notebook only configures,
calls, and plots.

**Panels produced here** (panels 1-2 - the annotated camera view + online
tracking/zones - are covered by a GUI screenshot, so they are not reproduced):

- **Panel 3 - outbound trajectories by cue.** ~20 consecutive trials at
  chance-level vs higher performance, each trace the cue->choice *outbound* run
  (return/ITI leg excluded), coloured by cue identity (cue A = high tone = blue,
  cue B = low tone = green). Expect intermixed colours at chance, cue-separated
  paths when performance is higher.
- **Panel 4 (VTE) - deliberation dwell.** Per-trial time in the central stem +
  junction (a vicarious-trial-and-error proxy), chance vs higher-performance
  block.
- **Optional** - a clean task-zone reference over the overhead view.

Default animal is **600** for the trajectory overlays; set `ANIMAL = "602"` and
re-run for the VTE-focused figure (602 deliberates more).

**Run:** `jupyter lab host/notebooks/grant_figures/proposal_figures.ipynb` from the
repo root in the `drieu_t_maze` env (after `pip install -e .`). Recordings are read
from `TMAZE_RECORDINGS_DIR` (default rig folder). Figures are written to `outputs/`.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from tmaze.analysis import bundle_reader, choice_signals
from tmaze.analysis import trajectories as tj
from tmaze.core import maze_state

# ----- configuration -------------------------------------------------------
ANIMAL = "600"          # 600 for the trajectory overlays; "602" for the VTE figure
START_DATE = "2026-08-24"
END_DATE = "2026-08-26"  # 8/26 bundles slot in automatically once recorded
BLOCK = 20               # trials per consecutive block
END_PAD_S = 0.6          # seconds past choice_time so a trace enters the chosen arm

# cue A = HIGH tone, cue B = LOW tone (tj.CUE_A / tj.CUE_B).
CUE_COLORS = {tj.CUE_A: "tab:blue", tj.CUE_B: "tab:green"}
CUE_NAMES = {tj.CUE_A: "cue A (high tone)", tj.CUE_B: "cue B (low tone)"}

# Optional manual overrides. Set to (session_id, first_trial) matching a row of the
# blocks table below to pin a block instead of the automatic low/high pick.
LOW_OVERRIDE = None
HIGH_OVERRIDE = None

# Zone outlines drawn on the overlays (the full figure-8 vocabulary).
ZONE_OUTLINES = (
    maze_state.HUB, maze_state.CENTRAL_STEM, maze_state.JUNCTION,
    maze_state.LEFT_CHOICE, maze_state.RIGHT_CHOICE,
    maze_state.LEFT_RETURN, maze_state.RIGHT_RETURN,
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("animal", ANIMAL, "| dates", START_DATE, "->", END_DATE, "| block", BLOCK)

## Load trials and pick the chance / higher-performance blocks

Load every bundle for the animal over the date span, derive per-trial signals
(`correct`, cue side, epoch), then cut each session into consecutive blocks of
`BLOCK` completed trials and auto-pick the lowest- and highest-accuracy blocks.
The table shows every block so you can pin a different pair via `LOW_OVERRIDE` /
`HIGH_OVERRIDE` above.

Note: on a stretch of late-training sessions the "low" block may sit above true
chance; the panel titles print the actual accuracy, so the figure stays honest.
Pull in an earlier session (widen `START_DATE`) if a genuine chance block is
wanted.

In [ ]:
data = bundle_reader.load_animals((ANIMAL,), START_DATE, END_DATE)
signals = choice_signals.add_signals(data.trials)
print(f"{ANIMAL}: {len(data.sessions)} session(s), {len(signals)} trials")

blocks = tj.make_blocks(signals, block=BLOCK)
blocks_df = pd.DataFrame([
    {"session_id": b.session_id, "first_trial": b.start_trial,
     "last_trial": b.end_trial, "accuracy": round(b.accuracy, 3),
     "cue_A": b.n_a, "cue_B": b.n_b}
    for b in blocks
])
display(blocks_df)

def resolve_block(override, default):
    if override is None:
        return default
    sid, first = override
    for b in blocks:
        if b.session_id == sid and b.start_trial == first:
            return b
    raise ValueError(f"override {override!r} does not match any block above")

low_auto, high_auto = tj.select_blocks(signals, block=BLOCK)
low = resolve_block(LOW_OVERRIDE, low_auto)
high = resolve_block(HIGH_OVERRIDE, high_auto)
assert low is not None and high is not None, "no full block of BLOCK trials found"
print("LOW  ->", low.label)
print("HIGH ->", high.label)

## Panel 3 - outbound trajectories coloured by cue

Each trace is one trial's cue->choice run (`[trial_start, choice_time] + END_PAD_S`),
drawn over the session's clean background frame with the task zones outlined. Blue =
cue A (high tone), green = cue B (low tone). Left = chance-level block, right =
higher-performance block.

In [ ]:
def bundle_assets(block):
    """(bundle, sidecar, zone_map, background_image_or_None) for a block's session."""
    bundle = tj.find_bundle(block.session_id)
    sc = tj.load_sidecar(bundle)
    zm = tj.load_zone_map(bundle)
    bg_path = Path(bundle) / "video" / f"{block.session_id}_background.png"
    img = plt.imread(bg_path) if bg_path.exists() else None
    return bundle, sc, zm, img


def _frame_dims(sc, img):
    if img is not None:
        h, w = img.shape[:2]
        return int(w), int(h)
    return int(sc.width), int(sc.height)


def draw_zones(ax, zm, W, H):
    for z in zm.zones:
        if z.name not in ZONE_OUTLINES:
            continue
        xy = np.asarray(z.polygon, dtype=float)
        xs = np.append(xy[:, 0], xy[0, 0]) * W
        ys = np.append(xy[:, 1], xy[0, 1]) * H
        ax.plot(xs, ys, color="0.55", lw=1.0, alpha=0.8, zorder=1)


def plot_overlay(ax, block, title):
    _, sc, zm, img = bundle_assets(block)
    W, H = _frame_dims(sc, img)
    if img is not None:
        ax.imshow(img, cmap="gray", zorder=0)
    else:
        ax.set_xlim(0, W)
        ax.set_ylim(H, 0)
    draw_zones(ax, zm, W, H)

    win = tj.outbound_windows(block.trials, end_pad_s=END_PAD_S)
    labelled = set()
    for _, r in win.iterrows():
        seg = tj.slice_xy(sc, r["t_start"], r["t_end"])
        if seg.empty:
            continue
        cue = r["cue"]
        ax.plot(seg["track_smooth_cx"], seg["track_smooth_cy"],
                color=CUE_COLORS.get(cue, "0.3"), lw=1.3, alpha=0.7, zorder=2,
                label=CUE_NAMES.get(cue) if cue not in labelled else None)
        labelled.add(cue)

    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_overlay(
    axes[0], low,
    f"Chance-level: {low.accuracy * 100:.0f}% correct "
    f"(n={low.n_a + low.n_b}, A={low.n_a}/B={low.n_b})",
)
plot_overlay(
    axes[1], high,
    f"Higher performance: {high.accuracy * 100:.0f}% correct "
    f"(n={high.n_a + high.n_b}, A={high.n_a}/B={high.n_b})",
)
fig.suptitle(f"Rat {ANIMAL} - outbound choice trajectories by cue", fontsize=13)
fig.tight_layout()
out = OUTPUT_DIR / f"trajectories_{ANIMAL}.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
print("saved", out.resolve())

## Panel 4 (optional) - VTE deliberation dwell

Per-trial time the animal spends in the **central stem + junction** during the
outbound run - a simple vicarious-trial-and-error proxy (a pause at cue onset in
the stem, head-swinging at the choice point). Compared between the chance-level
and higher-performance blocks, with a SciPy-free Mann-Whitney U. Most informative
for rat 602 (set `ANIMAL = "602"` and re-run from the top).

In [ ]:
def dwell_for_block(block, group_label):
    """Per-trial VTE dwell for one block, tagged with a group label."""
    bundle = tj.find_bundle(block.session_id)
    sc = tj.load_sidecar(bundle)
    zm = tj.load_zone_map(bundle)
    win = tj.outbound_windows(block.trials, end_pad_s=END_PAD_S)
    dw = tj.dwell_per_trial(sc, zm, win)
    dw = dw[dw["n_samples"] > 0].copy()   # drop trials with no tracking in-window
    dw["group"] = group_label
    return dw

low_lbl = f"chance ({low.accuracy * 100:.0f}%)"
high_lbl = f"higher ({high.accuracy * 100:.0f}%)"
vte_low = dwell_for_block(low, low_lbl)
vte_high = dwell_for_block(high, high_lbl)

groups = [vte_low["dwell_s"].to_numpy(), vte_high["dwell_s"].to_numpy()]
labels = [low_lbl, high_lbl]
U, p = tj.mann_whitney_u(groups[0], groups[1])

fig, ax = plt.subplots(figsize=(5, 5))
ax.boxplot(groups, showfliers=False, widths=0.55)
rng = np.random.default_rng(0)
for i, g in enumerate(groups, start=1):
    ax.scatter(rng.normal(i, 0.05, len(g)), g, s=20, alpha=0.6, color="0.4", zorder=3)
ax.set_xticks([1, 2])
ax.set_xticklabels(labels)
ax.set_ylabel("time in central stem + junction (s)")
ax.set_title(
    f"Rat {ANIMAL} VTE dwell - Mann-Whitney U={U:.0f}, p={p:.3f}", fontsize=10
)
fig.tight_layout()
out = OUTPUT_DIR / f"vte_dwell_{ANIMAL}.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
print("saved", out.resolve())
for lbl, g in zip(labels, groups):
    print(f"  {lbl}: n={len(g)}  mean={g.mean():.2f}s  median={np.median(g):.2f}s")

## Optional - task-zone reference over the overhead view

A labelled fill of every task zone on the clean background frame. Handy as a
schematic beside the GUI screenshot for panels 1-2.

In [ ]:
ref = high  # any block; its session supplies the background + zones
_, sc, zm, img = bundle_assets(ref)
W, H = _frame_dims(sc, img)

fig, ax = plt.subplots(figsize=(6, 5))
if img is not None:
    ax.imshow(img, cmap="gray", zorder=0)
else:
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
cmap = plt.get_cmap("tab10")
for k, z in enumerate(zm.zones):
    xy = np.asarray(z.polygon, dtype=float) * [W, H]
    ax.add_patch(mpatches.Polygon(xy, closed=True, fill=True, alpha=0.25,
                                  color=cmap(k % 10), zorder=1))
    c = xy.mean(axis=0)
    ax.text(c[0], c[1], z.name.replace("_", "\n"), ha="center", va="center",
            fontsize=7, zorder=2)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
ax.set_title(f"Task zones on the overhead view ({ref.session_id})", fontsize=10)
fig.tight_layout()
out = OUTPUT_DIR / "zones_reference.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
print("saved", out.resolve())